# WhisperX Pipeline v6

Пайплайн транскрибации видео уроков с диаризацией спикеров и LLM-анализом.

**Этапы:**
1. Загрузка видео с Яндекс Диска
2. Шумоподавление (noisereduce)
3. Транскрибация (WhisperX large-v3 + Silero VAD)
4. Пост-обработка + повторная транскрибация проблемных зон (Pass 2)
5. Выравнивание слов (wav2vec2)
6. Диаризация спикеров (pyannote 3.1)
7. Аналитика (шум, темп, баланс, вовлечённость)
8. LLM-анализ (GigaChat: таймлайн + оценка качества)

Перед запуском настройте секреты: **Add-ons → Secrets** → HF_TOKEN, GIGACHAT_CREDENTIALS, YANDEX_TOKEN

In [ ]:
import subprocess, shutil, os, sys

# Удаляем старую копию репо если есть
REPO_DIR = "/kaggle/working/whisperx-pipeline"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("🗑️ Старая копия удалена")

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

# ── Шаг 1: whisperx без torch-зависимостей (используем Kaggle torch) ──
print("📦 Шаг 1/5: WhisperX (без torch)...")
pip("whisperx", "--no-deps")
pip("faster-whisper", "--no-deps")
pip("ctranslate2", "av", "tokenizers>=0.13")

# ── Шаг 2: pyannote ──
print("📦 Шаг 2/5: Pyannote...")
pip("pyannote.audio", "--no-deps")
pip("pyannote.core", "pyannote.database", "pyannote.metrics", "pyannote.pipeline")
pip("speechbrain", "asteroid-filterbanks", "torch-audiomentations", "einops", "lightning")

# ── Шаг 3: transformers (совместимая версия) ──
print("📦 Шаг 3/5: Transformers...")
pip("transformers>=4.40,<4.52", "huggingface-hub>=0.20")

# ── Шаг 4: аналитика и утилиты ──
print("📦 Шаг 4/5: Утилиты...")
pip("noisereduce", "soundfile", "nltk", "pyyaml", "optuna", "hyperpyyaml", "docopt", "rich")

# ── Шаг 5: API клиенты ──
print("📦 Шаг 5/5: API...")
pip("yadisk", "gigachat")

print("✅ Все пакеты установлены")

# ── Клонируем репозиторий ──
print("📥 Клонирование репозитория...")
subprocess.run(["git", "clone",
    "https://github.com/Hipposum/whisperx-pipeline.git",
    REPO_DIR], check=True)
print("✅ Репозиторий клонирован")

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
s = UserSecretsClient()
os.environ["HF_TOKEN"] = s.get_secret("HF_TOKEN")
os.environ["GIGACHAT_CREDENTIALS"] = s.get_secret("GIGACHAT_CREDENTIALS")
os.environ["YANDEX_TOKEN"] = s.get_secret("YANDEX_TOKEN")

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "src.pipeline"],
    cwd="/kaggle/working/whisperx-pipeline"
)
if result.returncode != 0:
    print(f"❌ Пайплайн завершился с ошибкой (код {result.returncode})")